# 06. Depth Estimation - FastDepth_1

| 항목 | 내용 |
|---|---|
| 대상 모델 | `FastDepth_1.dxnn` |
| 카테고리 | **Depth Estimation** |
| 기준 환경 | **DX-All-Suite 2.3.0 / Sync 흐름** |

## 학습 목표

1. 출력 값의 크기와 거리의 관계를 실행 결과의 좌표별 값으로 확인해 설명할 수 있다.
2. 최소·최대로 펼치는 정규화가 절대 거리 정보를 없앤다는 사실을 계산으로 보일 수 있다.
3. 색으로 그린 깊이 그림을 서로 다른 사진끼리 비교할 수 없는 이유를 설명할 수 있다.
4. 결과 객체가 원시 값과 색 그림을 함께 담는 이유를 두 쓰임과 연결해 설명할 수 있다.

## 1. Depth Estimation (깊이 추정)

**한 장의 사진에서 픽셀마다 카메라로부터의 거리를 추정하는 작업**

사람은 두 눈의 차이로 거리를 재지만 사진은 한 장뿐이다. 이 작업은 크기·가려짐·원근·무늬의 촘촘함 같은
단서를 학습해 한 장에서 거리를 짐작한다.

**출력의 성격**

픽셀마다 값이 하나씩 나오는 지도이며, 값의 대소가 거리의 순서를 나타낸다. 두 눈이나 거리 센서 없이 짐작한
값이므로 절대 거리를 보증하지 않고, 학습에 없던 장면에서는 순서까지 어긋날 수 있다.

**활용 사례**

- 배경 흐림 효과의 경계 판정
- 로봇·드론의 장애물 회피 보조
- 사진 한 장에서 대상까지의 순서 판단

**포함되지 않는 작업**

- 무엇이 있는지 판정 (Object Detection)
- 픽셀마다 종류 구분 (Semantic Segmentation)
- 거리를 센서로 측정 (LiDAR·ToF 같은 장치의 일)

## 2. 전체 파이프라인

```text
sample_horse.jpg  773×512 BGR
  → 전처리         BGR→RGB · 224×224 직접 축소 · uint8 유지
  → run()          입력 Tensor 하나를 NPU 에 전달
  → Public Output  output  [1, 1, 224, 224]  float32  (NCHW, 값 0.88~2.71)
  → 후처리         squeeze → 최소·최대로 0~255 펼침 → MAGMA colormap → 원본 크기
  → Result         DepthResult (원시 깊이 지도 + 색으로 그린 지도)
  → 시각화         색으로 그린 깊이 지도를 원본 크기로 표시
```

![06. Depth Estimation 전체 파이프라인](resources/06_Depth_Estimation_pipeline.png)

## 3. 입력과 출력

### 3.1 입력

| 속성 | 값 |
|---|---|
| Tensor 이름 | `input` |
| Shape / Dtype | `[1, 224, 224, 3]` / `uint8` |
| 채널 순서 | NHWC |
| Color / 값 범위 | RGB / 0~255 |
| Resize 방식 | 직접 축소 (비율 유지 없음) |
| Normalization | 없음 |

**다른 모델과의 차이점**

- **입력은 컬러, 출력은 한 채널** — 색 세 장이 들어가 거리 한 장이 나온다
- **비율을 유지하지 않음** — 773×512 가 224×224 로 눌리므로 가로가 더 많이 줄어든다. 거리의 순서를 보는
  용도라 좌표를 되돌릴 일이 없어 그대로 쓴다

### 3.2 출력

| Tensor | Shape / dtype | 의미 | Public |
|---|---|---|---|
| `output` | `[1, 1, 224, 224]` / `float32` | 픽셀마다의 거리 추정값 | O |

`Public` 은 `InferenceEngine.run()` 이 돌려주는 값인지를 가리키는 표시다. 이 모델은 Task 가 `npu_0` 하나뿐이라
NPU 가 계산한 값이 그대로 나온다.

**shape 각 자리의 뜻**

| index | 값 | 뜻 |
|---|---:|---|
| 0 | 1 | 한 번에 처리하는 사진 장수 |
| 1 | 1 | 채널. 거리 한 장 |
| 2 | 224 | 높이 |
| 3 | 224 | 너비 |

입력은 NHWC 인데 출력은 NCHW 다. 채널이 1개라 축을 옮겨도 값의 순서가 바뀌지 않으므로 후처리는 크기 1인 축을
눌러 없애는 것으로 끝낸다. `float32` 하나가 4바이트이므로 `224 × 224` 는 200,704 바이트다.

**값이 클수록 멀다.** 이번 실행에서 사진 아래쪽 가까운 바닥은 `0.94`, 위쪽 먼 배경은 `2.17` 로 나왔다. 값의
대소가 거리의 순서를 그대로 나타낸다.

**단위는 확인되지 않았다.** 값의 범위가 `0.88`~`2.71` 인데 이 사진에는 수십 미터 뒤의 배경까지 담겨 있어
실제 거리로 보기 어렵고, 코드에도 단위를 다루는 부분이 없다. 그래서 이 노트북은 **거리의 순서까지만 다루고
값에 미터 같은 단위를 붙이지 않는다.** "저쪽이 더 멀다"는 말할 수 있고 "저것은 8미터 앞"은 말할 수 없다.

#### 출력을 사람이 읽는 값으로 바꾸기

거리 값은 화면에 그릴 수 없으므로 **그 사진의 최솟값과 최댓값을 0 과 255 에 맞춰 펼친** 뒤 색을 입힌다.

```text
표시값 = (거리 - 최솟값) / (최댓값 - 최솟값) × 255
```

이번 실행의 최솟값 `0.8799`, 최댓값 `2.7067` 을 넣으면 평균 `1.9006` 은
`(1.9006 - 0.8799) / 1.8268 × 255 = 142.5` 이고, `uint8` 로 바뀌며 소수점이 버려져 142 가 된다.

**이 정규화가 절대 거리 정보를 없앤다.** 최솟값이 0, 최댓값이 255 로 고정되므로 **어느 사진이든 가장 가까운
곳은 같은 색, 가장 먼 곳도 같은 색이 된다.** 가까운 것만 찍은 사진과 멀리까지 찍은 사진이 똑같은 색 분포로
그려지므로, **색으로 두 사진의 거리를 비교하면 틀린다.** 색은 한 장 안에서의 순서를 보는 용도다.

색은 MAGMA 를 쓴다. 어두운 자리에서 밝은 자리로 단조롭게 밝아지는 색표라 순서가 눈에 잘 남고, 색맹인
사람에게도 순서가 유지된다. 색표를 바꾸면 그림만 달라지고 값은 그대로다.

#### `.dxnn` 파일에서 직접 확인

`parse_model` 은 `.dxnn` 안의 tensor 이름·shape·dtype·크기·Task 구성을 그대로 보고한다. 입력이 3채널 NHWC,
출력이 1채널 NCHW 인지 아래 출력에서 확인한다.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "assets" / "models").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
MODEL_PATH = REPO_ROOT / "assets" / "models" / "FastDepth_1.dxnn"
assert MODEL_PATH.is_file(), f"모델이 없다: {MODEL_PATH}"

os.environ.setdefault("DX_ALL_SUITE", os.path.expanduser("~/Workspace/dx-all-suite"))
DX_ALL_SUITE = Path(os.environ["DX_ALL_SUITE"])

DX_APP = DX_ALL_SUITE / "dx-runtime" / "dx_app"
assert DX_APP.is_dir(), f"dx-all-suite 가 없다: {DX_APP} — 환경변수 DX_ALL_SUITE 로 위치를 알려 준다."

parsed = subprocess.run(["parse_model", "-m", str(MODEL_PATH), "-v"], capture_output=True, text=True)
print((parsed.stdout + parsed.stderr).strip())

## 4. 실습

### 4.1 실행 환경 구성

라이브러리·그림 출력 방식·입력 파일 경로를 정한다. 이후 셀은 여기서 만든 이름을 그대로 쓴다.

In [ ]:
import sys
import time
from dataclasses import dataclass, field

import matplotlib

# 노트북에서는 결과 그림이 셀 아래 남도록 inline, 스크립트로 단독 실행할 때는 화면이 없으므로
# 파일 기반 Agg 를 쓴다. 지정하지 않으면 GUI 백엔드를 찾다가 실패한다.
try:
    get_ipython().run_line_magic("matplotlib", "inline")  # noqa: F821
except NameError:
    matplotlib.use("Agg")

# 렌더링 백엔드를 dx_engine 보다 먼저 적재한다. matplotlib 은 첫 그림을 그릴 때에야 백엔드를
# 불러오는데, 그 시점이 dx_engine 적재 뒤이면 ft2font 확장 초기화가 실패한다
# (RuntimeError: module 'matplotlib.ft2font' has no attribute '__path__').
import matplotlib.backends.backend_agg  # noqa: E402,F401
import matplotlib.pyplot as plt  # noqa: E402

import cv2  # noqa: E402
import numpy as np  # noqa: E402
from dx_engine import InferenceEngine  # noqa: E402

sys.dont_write_bytecode = True
IMAGE_PATH = DX_APP / "sample" / "img/sample_horse.jpg"

print("input image:", IMAGE_PATH.name)

#### 결과 객체와 실행 지원 함수

- `PreprocessContext` — 전처리가 원본을 어떻게 바꾸었는지 후처리에 넘기는 값. 후처리가 색 그림을 원본 크기로
  되돌리는 데 원본 크기를 쓴다
- `DepthResult` — 후처리가 만들어 시각화로 넘기는 결과. **원시 거리 값과 색 그림을 함께** 담는다. 사람이 볼
  그림과 뒤 단계가 계산에 쓸 값이 다르기 때문이다
- `notebook_describe_array` — 배열의 shape·dtype·값 범위 출력
- `notebook_show_bgr` — 이미지 inline 표시. 3채널은 RGB 로 되돌림(안 하면 색이 뒤집힘), grayscale 은
  값 범위를 0~255 로 고정(안 하면 대비가 실제보다 강해짐)

In [ ]:
@dataclass
class PreprocessContext:
    """전처리가 원본을 어떻게 바꾸었는지 후처리에 넘기는 값. 원본 대응: common/base/i_processor.py"""

    original_width: int = 0
    original_height: int = 0
    input_width: int = 0
    input_height: int = 0
    scale_x: float = 0.0
    scale_y: float = 0.0
    pad_x: int = 0
    pad_y: int = 0


@dataclass
class DepthResult:
    """원시 깊이 지도와 색으로 그린 지도. 원본 대응: common/processors/depth_postprocessor.py"""

    depth_map: np.ndarray = field(default=None)
    depth_colormap: np.ndarray = field(default=None)


def notebook_describe_array(name: str, array: np.ndarray) -> None:
    """배열의 shape·dtype·값 범위를 한 줄로 출력한다. 노트북 전용."""
    array = np.asarray(array)
    print(f"{name:16s} {str(array.shape):18s} {array.dtype!s:8s} "
          f"min={array.min():9.4f} max={array.max():9.4f} mean={array.mean():9.4f}")


def notebook_show_bgr(image: np.ndarray, title: str = "Image", figsize: tuple = (9, 6)) -> None:
    """BGR 또는 grayscale 이미지를 inline 으로 표시한다. 노트북 전용."""
    plt.figure(figsize=figsize)
    if image.ndim == 2:
        plt.imshow(image, cmap="gray", vmin=0, vmax=255)
    else:
        plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

#### 런타임이 보고하는 입출력 정보

`.dxnn` 을 NPU 에 올리고 런타임이 보고하는 입출력 정보를 읽는다. `parse_model` 출력과 같아야 한다.

In [ ]:
engine = InferenceEngine(str(MODEL_PATH))
input_info = engine.get_input_tensors_info()
output_info = engine.get_output_tensors_info()

print("input :", input_info)
print("output:", output_info)

INPUT_HEIGHT, INPUT_WIDTH = input_info[0]["shape"][1], input_info[0]["shape"][2]
print(f"model input (WxH) = {INPUT_WIDTH}x{INPUT_HEIGHT}")

### 4.2 전처리

BGR 원본을 RGB 로 뒤집고 224×224 로 맞춘다. 비율을 유지하지 않아 가로가 세로보다 더 많이 줄어들며, 거리의
순서를 보는 용도라 그 눌림을 되돌리지 않는다.

확인할 것:

- 색 순서 — BGR → RGB
- 크기 — 224×224
- dtype — `uint8` 유지
- 축소 비율 — 가로와 세로가 다름

#### 입력 이미지 확인

원본 사진을 읽어 크기와 dtype 을 확인한다.

In [ ]:
image = cv2.imread(str(IMAGE_PATH))

print(f"shape {image.shape} dtype {image.dtype}")
notebook_show_bgr(image, f"Input - {IMAGE_PATH.name}")

#### 전처리 함수 정의

- `fastdepth_preprocessor_create` — 모델 입력 크기를 state 에 담음
- `fastdepth_preprocessor_process` — 이미지 한 장을 입력 Tensor 와 Context 로 변환

In [ ]:
def fastdepth_preprocessor_create(input_width: int, input_height: int) -> dict:
    """모델 입력 크기와 고정 상수를 state 에 담는다. 원본 대응: Factory.create_preprocessor()"""
    return {"input_width": input_width, "input_height": input_height}


def fastdepth_preprocessor_process(state: dict, input_image: np.ndarray) -> tuple:
    """이미지를 입력 Tensor 와 Context 로 바꾼다. 원본 대응: Runner.preprocess()"""
    ctx = PreprocessContext()
    ctx.original_height, ctx.original_width = input_image.shape[:2]
    ctx.input_width = state["input_width"]
    ctx.input_height = state["input_height"]
    ctx.scale_x = float(ctx.input_width) / float(ctx.original_width)
    ctx.scale_y = float(ctx.input_height) / float(ctx.original_height)

    rgb = cv2.cvtColor(input_image, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, (ctx.input_width, ctx.input_height),
                         interpolation=cv2.INTER_LINEAR)
    return resized, ctx

#### 전처리 실행

입력 Tensor 와 Context 를 만든다. `scale_x` 가 `scale_y` 보다 작은 것이 가로가 더 많이 줄었다는 뜻이다.

In [ ]:
preprocessor_state = fastdepth_preprocessor_create(INPUT_WIDTH, INPUT_HEIGHT)
input_tensor, ctx = fastdepth_preprocessor_process(preprocessor_state, image)

print(f"input_tensor  shape={input_tensor.shape} dtype={input_tensor.dtype} "
      f"contiguous={input_tensor.flags['C_CONTIGUOUS']}")
print(f"context       original={ctx.original_width}x{ctx.original_height} "
      f"scale_x={ctx.scale_x:.4f} scale_y={ctx.scale_y:.4f}")

### 4.3 추론

#### DXNN Inference 실행

입력 Tensor 를 NPU 에 넘기고 Public Output 을 받는다. 출력되는 시간은 1회 측정값이고 런타임과의 왕복
비용이 포함돼 있어 초당 처리 장수로 환산하지 않는다.

In [ ]:
started = time.perf_counter()
outputs = engine.run([input_tensor])
inference_ms = (time.perf_counter() - started) * 1000.0

print(f"inference: {inference_ms:.2f} ms")
print("output count =", len(outputs))
for index, output in enumerate(outputs):
    array = np.asarray(output)
    print(index, array.shape, array.dtype, f"min={array.min():.4f}", f"max={array.max():.4f}")

#### Raw Output 직접 검사

값에서 직접 확인할 것:

- 값 범위 — 0~1 도 0~255 도 아닌 구간이라 그대로 그릴 수 없다
- 좌표별 값 — 가까운 바닥과 먼 배경 중 어느 쪽이 큰지
- 백분위 — 값이 좁은 구간에 몰려 있는지

In [ ]:
raw = np.asarray(outputs[0])
notebook_describe_array("raw output", raw)

depth = np.squeeze(raw)
height, width = depth.shape
print(f"squeeze 뒤 shape = {depth.shape}")
print(f"  아래쪽 바닥 (y={height - 10}, x={width // 2}) = {depth[height - 10, width // 2]:.4f}")
print(f"  위쪽 배경   (y=10, x=10)      = {depth[10, 10]:.4f}")
print(f"  가운데      (y={height // 2}, x={width // 2}) = {depth[height // 2, width // 2]:.4f}")
for percent in (5, 50, 95):
    print(f"  {percent:2d}분위 = {np.percentile(depth, percent):.4f}")

### 4.4 후처리

크기 1인 축을 눌러 `[224, 224]` 로 만들고, 그 사진의 최소·최대로 0~255 에 펼친 뒤 색을 입혀 원본 크기로
되돌린다. 순서는 squeeze → 정규화 → colormap → resize 이며 이 순서를 바꾸지 않는다.

**원시 값과 색 그림을 모두 Result 에 담는 것이 이 후처리의 특징이다.** 색 그림은 사람이 보는 용도이고, 뒤에서
거리를 계산에 쓰려면 정규화되지 않은 원시 값이 필요하다. 색 그림만 남기면 절대 크기 정보를 되찾을 수 없다.

색표 기본값은 MAGMA 이며 설정으로 바꿀 수 있다. 이 모델은 설정 파일이 비어 있어 후처리 코드의 기본값이
그대로 쓰인다.

#### 후처리 함수 정의

- `fastdepth_postprocessor_create` — 색표를 state 에 담음
- `fastdepth_normalize` — 최소·최대로 0~255 에 펼침
- `fastdepth_postprocessor_process` — 정규화·색 입히기·원본 크기 복원을 거쳐 Result 생성

In [ ]:
def fastdepth_postprocessor_create(colormap: int = cv2.COLORMAP_MAGMA) -> dict:
    """고정 상수를 state 에 담는다. 원본 대응: Factory.create_postprocessor()"""
    return {"colormap": colormap}


def fastdepth_normalize(depth: np.ndarray) -> np.ndarray:
    """그 사진의 최소·최대를 0~255 에 맞춰 펼친다. 원본 대응: DepthEstimationPostprocessor.process()"""
    d_min, d_max = depth.min(), depth.max()
    if d_max - d_min <= 1e-6:
        return np.zeros_like(depth, dtype=np.uint8)
    return ((depth - d_min) / (d_max - d_min) * 255).astype(np.uint8)


def fastdepth_postprocessor_process(state: dict, outputs: list, ctx: PreprocessContext) -> list:
    """Raw Output 을 Result 로 변환한다. 원본 대응: Runner.postprocess()"""
    depth = np.squeeze(outputs[0])
    depth_color = cv2.applyColorMap(fastdepth_normalize(depth), state["colormap"])
    depth_color = cv2.resize(depth_color, (ctx.original_width, ctx.original_height))
    return [DepthResult(depth_map=depth, depth_colormap=depth_color)]

#### 후처리 실행

확인할 것:

- `depth_map` — 원시 값이 그대로 남아 있는지
- `depth_colormap` — `uint8` 3채널이고 원본 크기인지
- 정규화 결과 — 최솟값이 0, 최댓값이 255 로 고정됐는지

In [ ]:
postprocessor_state = fastdepth_postprocessor_create()
results = fastdepth_postprocessor_process(postprocessor_state, outputs, ctx)
result = results[0]

notebook_describe_array("depth_map", result.depth_map)
print(f"depth_colormap  shape={result.depth_colormap.shape} dtype={result.depth_colormap.dtype}")

normalized = fastdepth_normalize(result.depth_map)
print(f"정규화 결과     min={normalized.min()} max={normalized.max()} mean={normalized.mean():.2f}")

d_min, d_max = float(result.depth_map.min()), float(result.depth_map.max())
mean_value = float(result.depth_map.mean())
print(f"평균값 변환     {mean_value:.4f} → "
      f"({mean_value:.4f} - {d_min:.4f}) / {d_max - d_min:.4f} × 255 = "
      f"{(mean_value - d_min) / (d_max - d_min) * 255:.1f}")

### 4.5 시각화

#### Visualizer 코드

결과 객체의 색 그림만 받아 원본 크기로 맞춰 돌려준다. 원본과 나란히 붙이지 않는 것은 화면 도구가 좌우 비교를
따로 제공하기 때문이며, 노트북에서는 다음 셀에서 직접 나란히 놓는다.

In [ ]:
def fastdepth_visualizer_visualize(image: np.ndarray, results: list) -> np.ndarray:
    """색으로 그린 깊이 지도를 원본 크기로 돌려준다. 원본 대응: Runner.visualize()"""
    height, width = image.shape[:2]
    return cv2.resize(results[0].depth_colormap, (width, height))

#### 결과 시각화

색으로 그린 깊이 지도를 inline 으로 표시한다. 밝은 쪽이 값이 큰 자리이며 이 사진에서는 먼 배경이다.

In [ ]:
output_image = fastdepth_visualizer_visualize(image, results)
notebook_show_bgr(output_image, "Depth colormap (MAGMA)", figsize=(10, 7))

#### 원본과 나란히 놓고 대조

같은 자리끼리 색과 원본을 대조하면 모델이 무엇을 가깝다고 보았는지 확인된다. 아래쪽 바닥이 어둡고 위쪽
배경이 밝게 나오는지 보면 값과 거리의 관계가 눈으로도 확인된다.

In [ ]:
side_by_side = np.hstack([image, output_image])
cv2.putText(side_by_side, "Input", (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 2)
cv2.putText(side_by_side, "Depth", (image.shape[1] + 10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2,
            (255, 255, 255), 2)
notebook_show_bgr(side_by_side, "Input vs Depth", figsize=(14, 5))